# Creating nice bubble plots of selected genes based on EdgeR statistics results (CM hypoxia+ FB activation)

In [2]:
# Configs and imports

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch
from mpl_toolkits.axes_grid1 import make_axes_locatable

# Custom gradient: dark blue -> light blue -> light orange-ish red -> dark orange
colors = ["#00008B", "#ADD8E6",  "#ffcc80", "#d35400"]
cmap = mcolors.LinearSegmentedColormap.from_list("custom_xeno", colors)

warnings.filterwarnings("ignore")
mpl.rcParams["pdf.fonttype"]    = 42
mpl.rcParams["ps.fonttype"]     = 42
mpl.rcParams["font.family"]     = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial"]

ANALYSIS_LEVEL   = "cell_state"       # "cell_type" or "cell_state"
REF_CONDITION    = "Control"  # "Control" or "3month-protocol"
TARGET_CELL_TYPE = "CM"               # Only for cell_state level

# Conditions to plot (one dotplot per condition)
CONDITIONS_TO_PLOT = [
    "6month-protocol",
    # "Prematurely-terminated",
    "3month-protocol",
    # "PCMV/PRV",
    "Rejection",
]

# Genes of interest 

GENES_OF_INTEREST = [
    "Sus_scrofa___HIF1A",  
    "Sus_scrofa___VEGFA",
    "Sus_scrofa___PFKFB3",  
    "Sus_scrofa___BNIP3", 
    "Sus_scrofa___PDK1",
    "Sus_scrofa___ENO1",  
]

# Preferred fallback order if a gene is entered without a prefix and multiple species exist
PREFERRED_SPECIES_ORDER = ["Sus_scrofa___", "Papio_anubis___", "pCMV_____"]

# Prefixes & paths
PREFIXES   = ["Sus_scrofa___", "Papio_anubis_", "pCMV_____"]
EDGER_BASE = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1"
RESULTS_DIR = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/fancy_bubbleplots/CM_hypoxia_markers"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Dot visual parameters
DOT_SIZE_MIN       = 7         
DOT_SIZE_MAX       = 320       
LOGFC_CLIP         = 3.0        # Symmetric color range [-clip, +clip]
COLORMAP           = cmap

# Significance border styles
BORDER_NS_COLOR    = "#CCCCCC"
BORDER_NS_WIDTH    = 0.5
BORDER_FDR05_COLOR = "black"
BORDER_FDR05_WIDTH = 1
BORDER_FDR01_COLOR = "black"
BORDER_FDR01_WIDTH = 2.2

#fig sizes
CELL_WIDTH         = 0.001      # Width per gene column (inches)
CELL_HEIGHT        = 0.23       # Height per entity row (inches)
LEGEND_EXTRA_WIDTH = 2.6        # Extra width for legends on the right (inches)
TOP_MARGIN_INCHES  = 1.2        # Extra top for rotated gene labels
BOTTOM_MARGIN      = 0.4        # Bottom margin (inches)

# font parameters
GENE_FONT_SIZE     = 9
GENE_ROTATION      = 90         # Rotation angle for gene labels
ROW_FONT_SIZE      = 10
ROW_FONT_WEIGHT    = "bold"
TITLE_FONT_SIZE    = 10
TITLE_FONT_WEIGHT  = "bold"

# grid inside figures
GRID_VISIBLE       = True
GRID_COLOR         = "#F0F0F0"
GRID_WIDTH         = 0.0

DPI = 300

print("Configuration loaded.")


# Let's load edgeR results to fetch signifcances and statistics for visualization

def extract_prefix_and_symbol(full_gene_str):
    """Splits a full gene string into its prefix and clean symbol components."""
    for prefix in PREFIXES:
        if full_gene_str.startswith(prefix):
            return prefix, full_gene_str[len(prefix):]
    return "", full_gene_str

def normalise(s):
    return s.lower().replace("/", "_").replace("-", "_").replace(".", "_").replace(" ", "_")

def resolve_ref_dir(ref):
    return "Ref_control" if ref == "Control" else "Ref_3month"

def find_edger_csv(raw_dir, comp, ref, entity, level):
    target = normalise(f"{comp}_vs_{ref}")
    if not os.path.isdir(raw_dir):
        return None
    for folder in os.listdir(raw_dir):
        if not os.path.isdir(os.path.join(raw_dir, folder)):
            continue
        if normalise(folder) != target:
            continue
        fpath = os.path.join(raw_dir, folder)
        if level == "cell_type":
            for f in os.listdir(fpath):
                if f.endswith(".csv") and entity in f:
                    return os.path.join(fpath, f)
        elif level == "cell_state":
            for ct in os.listdir(fpath):
                ct_p = os.path.join(fpath, ct)
                if not os.path.isdir(ct_p):
                    continue
                for f in os.listdir(ct_p):
                    if f.endswith(".csv") and entity in f:
                        return os.path.join(ct_p, f)
    return None

# create a path
ref_dir = resolve_ref_dir(REF_CONDITION)
raw_dir = os.path.join(EDGER_BASE, ANALYSIS_LEVEL, ref_dir, "results", "edgeR_raw")
print(f"Reading from: {raw_dir}")

entities = set()
for comp_folder in os.listdir(raw_dir):
    if ANALYSIS_LEVEL == "cell_type":
        cp = os.path.join(raw_dir, comp_folder)
        if os.path.isdir(cp):
            for f in os.listdir(cp):
                if f.endswith(".csv"):
                    ct = f.replace("edgeR_Raw_", "").split(f"_{comp_folder}")[0]
                    entities.add(ct)
    elif ANALYSIS_LEVEL == "cell_state":
        cp = os.path.join(raw_dir, comp_folder, TARGET_CELL_TYPE)
        if os.path.isdir(cp):
            for f in os.listdir(cp):
                if f.endswith(".csv"):
                    cs = f.replace("edgeR_Raw_", "").split(f"_{comp_folder}")[0]
                    entities.add(cs)
entities = sorted(entities)
print(f"Entities: {entities}")

# Parse the desired genes into mapping dictionaries
gene_query_mapping = {}
display_genes_order = []

for item in GENES_OF_INTEREST:
    pref, sym = extract_prefix_sub = extract_prefix_and_symbol(item)
    if pref != "":
        display_label = sym  # Clean name shown on plot axis
        gene_query_mapping[display_label] = {"strict_full": item, "clean_symbol": sym}
    else:
        display_label = item
        gene_query_mapping[display_label] = {"strict_full": None, "clean_symbol": item}
    display_genes_order.append(display_label)

# Read CSVs and match specific requested genes/species
rows = []
for cond in CONDITIONS_TO_PLOT:
    for ent in entities:
        csv = find_edger_csv(raw_dir, cond, REF_CONDITION, ent, ANALYSIS_LEVEL)
        if csv is None:
            continue
        df = pd.read_csv(csv)
        gcol = [c for c in df.columns if c.lower() == "gene"]
        if not gcol:
            continue
        
        # Build lookup pool for this dataframe file
        df["raw_gene"] = df[gcol[0]].astype(str)
        
        for display_label in display_genes_order:
            rule = gene_query_mapping[display_label]
            strict_target = rule["strict_full"]
            clean_sym = rule["clean_symbol"]
            
            matched_row = None
            if strict_target is not None:
               
                sub = df[df["raw_gene"] == strict_target]
                if not sub.empty:
                    matched_row = sub.iloc[0]
            else:
                candidates = df[df["raw_gene"].apply(lambda x: extract_prefix_and_symbol(x)[1] == clean_sym)]
                if not candidates.empty:

                    def get_priority(val):
                        pref, _ = extract_prefix_and_symbol(val)
                        if pref in PREFERRED_SPECIES_ORDER:
                            return PREFERRED_SPECIES_ORDER.index(pref)
                        return 999
                    
                    candidates["priority"] = candidates["raw_gene"].apply(get_priority)
                    matched_row = candidates.sort_values("priority").iloc[0]

            if matched_row is not None:
                rows.append({
                    "Condition": cond, 
                    "Entity": ent, 
                    "Gene": display_label,  # Stored cleanly without prefix for the plotting logic
                    "logFC": matched_row.get("logFC", np.nan), 
                    "FDR": matched_row.get("FDR", np.nan),
                })

master = pd.DataFrame(rows)
master["abs_logFC"] = master["logFC"].abs()
master.round(4).to_csv(os.path.join(RESULTS_DIR, "bubble_master_data.csv"), index=False)
print(f"Master: {len(master)} rows generated successfully.")


# plotting
master = pd.read_csv(os.path.join(RESULTS_DIR, "bubble_master_data.csv"))
master["abs_logFC"] = master["logFC"].abs()

# Only show genes that are found in the lists

genes_present = [g for g in display_genes_order if g in master["Gene"].unique()]
entity_order  = sorted(master["Entity"].unique())
global_max    = max(master["abs_logFC"].dropna().max(), 0.5)

cmap = plt.cm.get_cmap(COLORMAP)
norm = mcolors.TwoSlopeNorm(vmin=-LOGFC_CLIP, vcenter=0, vmax=LOGFC_CLIP)

def scale_size(absfc):
    frac = min(absfc / global_max, 1.0)
    return DOT_SIZE_MIN + frac * (DOT_SIZE_MAX - DOT_SIZE_MIN)

for cond in CONDITIONS_TO_PLOT:
    df_c = master[master["Condition"] == cond]
    if df_c.empty:
        continue

    ng = len(genes_present)
    nr = len(entity_order)

    fig_w = ng * CELL_WIDTH + LEGEND_EXTRA_WIDTH
    fig_h = nr * CELL_HEIGHT + TOP_MARGIN_INCHES + BOTTOM_MARGIN

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    # Grid
    if GRID_VISIBLE:
        for i in range(nr):
            ax.axhline(i, color=GRID_COLOR, linewidth=GRID_WIDTH, zorder=0)
        for j in range(ng):
            ax.axvline(j, color=GRID_COLOR, linewidth=GRID_WIDTH, zorder=0)

    # Dots
    for i, ent in enumerate(entity_order):
        for j, gene in enumerate(genes_present):
            row = df_c[(df_c["Entity"] == ent) & (df_c["Gene"] == gene)]
            if row.empty:
                continue
            lfc = row["logFC"].values[0]
            fdr = row["FDR"].values[0]
            afc = row["abs_logFC"].values[0]
            if np.isnan(lfc):
                continue

            color = cmap(norm(np.clip(lfc, -LOGFC_CLIP, LOGFC_CLIP)))
            sz    = scale_size(afc)

            if np.isnan(fdr) or fdr > 0.05:
                ec, ew = BORDER_NS_COLOR, BORDER_NS_WIDTH
            elif fdr <= 0.01:
                ec, ew = BORDER_FDR01_COLOR, BORDER_FDR01_WIDTH
            else:
                ec, ew = BORDER_FDR05_COLOR, BORDER_FDR05_WIDTH

            ax.scatter(j, i, s=sz, c=[color], edgecolors=ec, linewidths=ew, zorder=5)

    # Axes
    ax.set_xticks(range(ng))
    ax.set_xticklabels(genes_present, rotation=GENE_ROTATION, ha="left",
                       fontsize=GENE_FONT_SIZE, style="italic", fontweight="bold")
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")

    ax.set_yticks(range(nr))
    ax.set_yticklabels(entity_order, fontsize=ROW_FONT_SIZE, fontweight=ROW_FONT_WEIGHT)

    ax.set_xlim(-0.6, ng - 0.4)
    ax.set_ylim(nr - 0.4, -0.6)
    ax.tick_params(axis="both", length=0)

    # Remove box spines
    for spine in ["top", "bottom", "left", "right"]:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_linewidth(2.0)  
        ax.spines[spine].set_color("black")

    # Title
    tag = f" ({TARGET_CELL_TYPE} states)" if ANALYSIS_LEVEL == "cell_state" else ""
    ax.set_title(f"{cond} vs {REF_CONDITION}{tag}",
                 fontsize=TITLE_FONT_SIZE, fontweight=TITLE_FONT_WEIGHT, pad=15)

    # legends visualization parmeters
    # Colorbar
    cbar_ax = fig.add_axes([1, 0.38, 0.025, 0.35])  # [left, bottom, width, height]
    cb = mpl.colorbar.ColorbarBase(cbar_ax, cmap=cmap, norm=norm, orientation="vertical")
    cb.set_label("log₂FC", fontsize=8, fontweight="bold")
    cb.ax.tick_params(labelsize=7)
    cb.outline.set_visible(False)

    #  Size of the legend 
    size_ticks = [v for v in [1.2, 1.0, 2.0, 3.0] if v <= global_max * 1.2]
    if not size_ticks:
        size_ticks = [0.5, 1.0]

    size_handles = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor="#AAAAAA",
               markeredgecolor="black", markeredgewidth=0.5,
               markersize=np.sqrt(scale_size(v)) * 0.8,
               label=f"{v:.1f}", lw=0)
        for v in size_ticks
    ]
    size_leg = fig.legend(
        handles=size_handles, title="|log₂FC|",
        loc="upper right", bbox_to_anchor=(1.4, 0.98),
        fontsize=7, title_fontproperties={"size": 8, "weight": "bold"},
        frameon=True, fancybox=True, edgecolor="#CCCCCC",
        labelspacing=1.0, handletextpad=0.8,
    )

    # Significance legends
    sig_handles = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor="#AAAAAA",
               markeredgecolor=BORDER_NS_COLOR, markeredgewidth=BORDER_NS_WIDTH,
               markersize=10, lw=0, label="NS"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="#AAAAAA",
               markeredgecolor=BORDER_FDR05_COLOR, markeredgewidth=BORDER_FDR05_WIDTH,
               markersize=10, lw=0, label="FDR ≤ 0.05"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="#AAAAAA",
               markeredgecolor=BORDER_FDR01_COLOR, markeredgewidth=BORDER_FDR01_WIDTH,
               markersize=10, lw=0, label="FDR ≤ 0.01"),
    ]
    fig.legend(
        handles=sig_handles, title="Significance",
        loc="lower right", bbox_to_anchor=(1.3, 0.02),
        fontsize=7, title_fontproperties={"size": 8, "weight": "bold"},
        frameon=True, fancybox=True, edgecolor="#CCCCCC",
        labelspacing=0.8,
    )

    # Save
    safe_cond = cond.replace("/", "_").replace("-", "_")
    safe_ref  = REF_CONDITION.replace("-protocol", "")
    tag2 = f"{TARGET_CELL_TYPE}_states" if ANALYSIS_LEVEL == "cell_state" else "celltypes"
    fname = f"bubble_{tag2}_{safe_cond}_vs_{safe_ref}"

    plt.savefig(os.path.join(RESULTS_DIR, f"{fname}.pdf"), dpi=DPI, bbox_inches="tight", transparent=True)
    plt.close()
    print(f"  Saved: {fname}.pdf")

print(f"\nDone. Results in: {RESULTS_DIR}")


Configuration loaded.
Reading from: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_state/Ref_control/results/edgeR_raw
Entities: ['CM1', 'CM2', 'CM3', 'CM4', 'CM5', 'Hypoxia_CM', 'ISG_CM']


findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial


Master: 126 rows generated successfully.


findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because

  Saved: bubble_CM_states_6month_protocol_vs_Control.pdf


findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because

  Saved: bubble_CM_states_3month_protocol_vs_Control.pdf


findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because

  Saved: bubble_CM_states_Rejection_vs_Control.pdf

Done. Results in: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/fancy_bubbleplots/CM_hypoxia_markers


# FB activation

In [3]:
# Imports and configurations

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
warnings.filterwarnings("ignore")
mpl.rcParams["pdf.fonttype"]    = 42
mpl.rcParams["ps.fonttype"]     = 42
mpl.rcParams["font.family"]     = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial"]


colors = ["#00008B", "#ADD8E6",  "#ffcc80", "#d35400"]
cmap = mcolors.LinearSegmentedColormap.from_list("custom_xeno", colors)

# Analysis level 
ANALYSIS_LEVEL   = "cell_type"        
REF_CONDITION    = "3month-protocol"  
TARGET_CELL_TYPE = "FB"             

# Conditions to plot (X-axis, in this order) 
CONDITIONS_TO_PLOT = [
    "6month-protocol",
    # "Prematurely-terminated",
    "3month-protocol",
    # "PCMV/PRV",
    "Rejection",
]

# Genes to visualize
GENES_OF_INTEREST = [
    'POSTN', 'FAP', 'ACTA2', 'COL1A1', 'FN1', 'KIT', 'VCAN', 'SULF1', 'TNC', 'CD44', 'TGM2'
]

PREFERRED_SPECIES_ORDER = ["Sus_scrofa___", "Papio_anubis_", "pCMV_____"]

#  Prefixes & paths
PREFIXES   = ["Sus_scrofa___", "Papio_anubis_", "pCMV_____"]
EDGER_BASE = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1"
RESULTS_DIR = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/fancy_bubbleplots/FB_activation"
os.makedirs(RESULTS_DIR, exist_ok=True)

#  Dot visual parameters
DOT_SIZE_MIN       = 7          
DOT_SIZE_MAX       = 320        
LOGFC_CLIP         = 3.0        
COLORMAP           = cmap       

# Significance border styles
BORDER_NS_COLOR    = "#CCCCCC"
BORDER_NS_WIDTH    = 0.5
BORDER_FDR05_COLOR = "black"
BORDER_FDR05_WIDTH = 1.0
BORDER_FDR01_COLOR = "black"
BORDER_FDR01_WIDTH = 2.2

# figure sizes
CELL_WIDTH         = 0.1       
CELL_HEIGHT        = 0.2        # Height per Gene row (inches)
LEGEND_EXTRA_WIDTH = 1.4        
TOP_MARGIN_INCHES  = 1.2        
BOTTOM_MARGIN      = 0.4        

#  Font parameters
GENE_FONT_SIZE     = 10
COND_FONT_SIZE     = 10
COND_ROTATION      = 45         # Rotation for condition labels on X-axis
LABEL_FONT_WEIGHT  = "bold"
TITLE_FONT_SIZE    = 11

# Grids
GRID_VISIBLE       = True
GRID_COLOR         = "#F0F0F0"
GRID_WIDTH         = 0.8

DPI = 300

print("Configuration loaded.")


# Loading edgeR results

def extract_prefix_and_symbol(full_gene_str):
    for prefix in PREFIXES:
        if full_gene_str.startswith(prefix):
            return prefix, full_gene_str[len(prefix):]
    return "", full_gene_str

def normalise(s):
    return s.lower().replace("/", "_").replace("-", "_").replace(".", "_").replace(" ", "_")

def resolve_ref_dir(ref):
    return "Ref_control" if ref == "Control" else "Ref_3month"

def find_edger_csv(raw_dir, comp, ref, entity, level):
    target = normalise(f"{comp}_vs_{ref}")
    if not os.path.isdir(raw_dir):
        return None
    for folder in os.listdir(raw_dir):
        if not os.path.isdir(os.path.join(raw_dir, folder)):
            continue
        if normalise(folder) != target:
            continue
        fpath = os.path.join(raw_dir, folder)
        if level == "cell_type":
            for f in os.listdir(fpath):
                if f.endswith(".csv") and entity in f:
                    return os.path.join(fpath, f)
        elif level == "cell_state":
            for ct in os.listdir(fpath):
                ct_p = os.path.join(fpath, ct)
                if not os.path.isdir(ct_p):
                    continue
                for f in os.listdir(ct_p):
                    if f.endswith(".csv") and entity in f:
                        return os.path.join(ct_p, f)
    return None

# Build path
ref_dir = resolve_ref_dir(REF_CONDITION)
raw_dir = os.path.join(EDGER_BASE, ANALYSIS_LEVEL, ref_dir, "results", "edgeR_raw")
print(f"Reading from: {raw_dir}")

entities = set()
if os.path.exists(raw_dir):
    for comp_folder in os.listdir(raw_dir):
        if ANALYSIS_LEVEL == "cell_type":
            cp = os.path.join(raw_dir, comp_folder)
            if os.path.isdir(cp):
                for f in os.listdir(cp):
                    if f.endswith(".csv"):
                        ct = f.replace("edgeR_Raw_", "").split(f"_{comp_folder}")[0]
                        entities.add(ct)
        elif ANALYSIS_LEVEL == "cell_state":
            cp = os.path.join(raw_dir, comp_folder, TARGET_CELL_TYPE)
            if os.path.isdir(cp):
                for f in os.listdir(cp):
                    if f.endswith(".csv"):
                        cs = f.replace("edgeR_Raw_", "").split(f"_{comp_folder}")[0]
                        entities.add(cs)

# ONLY keep the targeted cell type
entities = [e for e in sorted(entities) if e == TARGET_CELL_TYPE]
print(f"Filtered Entities for Plotting: {entities}")

if not entities:
    print(f"ERROR: Target '{TARGET_CELL_TYPE}' not found in edgeR results.")

# Parse requested genes into mapping dictionaries
gene_query_mapping = {}
display_genes_order = []

for item in GENES_OF_INTEREST:
    pref, sym = extract_prefix_and_symbol(item)
    if pref != "":
        display_label = sym  # Clean name shown on plot axis
        gene_query_mapping[display_label] = {"strict_full": item, "clean_symbol": sym}
    else:
        display_label = item
        gene_query_mapping[display_label] = {"strict_full": None, "clean_symbol": item}
    display_genes_order.append(display_label)

# Read CSVs
rows = []
for cond in CONDITIONS_TO_PLOT:
    for ent in entities:
        csv = find_edger_csv(raw_dir, cond, REF_CONDITION, ent, ANALYSIS_LEVEL)
        if csv is None:
            continue
        df = pd.read_csv(csv)
        gcol = [c for c in df.columns if c.lower() == "gene"]
        if not gcol:
            continue
            
        df["raw_gene"] = df[gcol[0]].astype(str)
        
        for display_label in display_genes_order:
            rule = gene_query_mapping[display_label]
            strict_target = rule["strict_full"]
            clean_sym = rule["clean_symbol"]
            
            matched_row = None
            if strict_target is not None:
                sub = df[df["raw_gene"] == strict_target]
                if not sub.empty:
                    matched_row = sub.iloc[0]
            else:
                candidates = df[df["raw_gene"].apply(lambda x: extract_prefix_and_symbol(x)[1] == clean_sym)]
                if not candidates.empty:
                    def get_priority(val):
                        pref, _ = extract_prefix_and_symbol(val)
                        if pref in PREFERRED_SPECIES_ORDER:
                            return PREFERRED_SPECIES_ORDER.index(pref)
                        return 999
                    
                    candidates_with_priority = candidates.copy()
                    candidates_with_priority["priority"] = candidates_with_priority["raw_gene"].apply(get_priority)
                    matched_row = candidates_with_priority.sort_values("priority").iloc[0]

            if matched_row is not None:
                rows.append({
                    "Condition": cond, 
                    "Entity": ent, 
                    "Gene": display_label, 
                    "logFC": matched_row.get("logFC", np.nan), 
                    "FDR": matched_row.get("FDR", np.nan),
                })

master = pd.DataFrame(rows)
if not master.empty:
    master["abs_logFC"] = master["logFC"].abs()
    master.round(4).to_csv(os.path.join(RESULTS_DIR, f"bubble_master_{TARGET_CELL_TYPE}_data.csv"), index=False)
    print(f"Master CSV Saved: {len(master)} rows for {TARGET_CELL_TYPE}")
else:
    print("Warning: No data found matching your genes and conditions.")


# Plotting section ------------------

if not master.empty:
    master["abs_logFC"] = master["logFC"].abs()
    global_max    = max(master["abs_logFC"].dropna().max(), 0.5)
    norm          = mcolors.TwoSlopeNorm(vmin=-LOGFC_CLIP, vcenter=0, vmax=LOGFC_CLIP)

    def scale_size(absfc):
        frac = min(absfc / global_max, 1.0)
        return DOT_SIZE_MIN + frac * (DOT_SIZE_MAX - DOT_SIZE_MIN)

    # We loop over entities (which is exclusively FB based on our filter)
    for ent in entities:
        df_ent = master[master["Entity"] == ent]
        if df_ent.empty:
            continue

        genes_present = [g for g in display_genes_order if g in df_ent["Gene"].unique()]
        conds_present = [c for c in CONDITIONS_TO_PLOT if c in df_ent["Condition"].unique()]

        ng = len(conds_present) # Number of columns (X-axis)
        nr = len(genes_present) # Number of rows (Y-axis)

        fig_w = ng * CELL_WIDTH + LEGEND_EXTRA_WIDTH
        fig_h = nr * CELL_HEIGHT + TOP_MARGIN_INCHES + BOTTOM_MARGIN

        fig, ax = plt.subplots(figsize=(fig_w, fig_h))

        # Grid
        if GRID_VISIBLE:
            for i in range(nr):
                ax.axhline(i, color=GRID_COLOR, linewidth=GRID_WIDTH, zorder=0)
            for j in range(ng):
                ax.axvline(j, color=GRID_COLOR, linewidth=GRID_WIDTH, zorder=0)

        # Dots mapping: (X=cond, Y=gene)
        for i, gene in enumerate(genes_present):
            for j, cond in enumerate(conds_present):
                row = df_ent[(df_ent["Condition"] == cond) & (df_ent["Gene"] == gene)]
                if row.empty:
                    continue
                
                lfc = row["logFC"].values[0]
                fdr = row["FDR"].values[0]
                afc = row["abs_logFC"].values[0]
                
                if np.isnan(lfc):
                    continue

                color = COLORMAP(norm(np.clip(lfc, -LOGFC_CLIP, LOGFC_CLIP)))
                sz    = scale_size(afc)

                if np.isnan(fdr) or fdr > 0.05:
                    ec, ew = BORDER_NS_COLOR, BORDER_NS_WIDTH
                elif fdr <= 0.01:
                    ec, ew = BORDER_FDR01_COLOR, BORDER_FDR01_WIDTH
                else:
                    ec, ew = BORDER_FDR05_COLOR, BORDER_FDR05_WIDTH

                ax.scatter(j, i, s=sz, c=[color], edgecolors=ec, linewidths=ew, zorder=5)

        # X-axis (Conditions)
        ax.set_xticks(range(ng))
        ax.set_xticklabels(conds_present, rotation=COND_ROTATION, ha="left",
                           fontsize=COND_FONT_SIZE, fontweight=LABEL_FONT_WEIGHT)
        ax.xaxis.tick_top()
        ax.xaxis.set_label_position("top")

        # Y-axis (Genes)
        ax.set_yticks(range(nr))
        ax.set_yticklabels(genes_present, fontsize=GENE_FONT_SIZE, style="italic", fontweight=LABEL_FONT_WEIGHT)

        ax.set_xlim(-0.4, ng - 0.6)
        ax.set_ylim(nr - 0.4, -0.6) # Flips Y-axis so first gene is at the top
        ax.tick_params(axis="both", length=0)

        # Enable and thicken the box spines (Bold border around plot)
        for spine in ["top", "bottom", "left", "right"]:
            ax.spines[spine].set_visible(True)
            ax.spines[spine].set_linewidth(2.0)  
            ax.spines[spine].set_color("black")

        # Title
        ax.set_title(f"{ent} Activation Profiling (vs {REF_CONDITION})",
                     fontsize=TITLE_FONT_SIZE, fontweight=LABEL_FONT_WEIGHT, pad=25)

        # Legends ----------------------------

        # Colorbar
        # Pulled closer to the plot
        cbar_ax = fig.add_axes([0.98, 0.3, 0.07, 0.35])  # [left, bottom, width, height]
        cb = mpl.colorbar.ColorbarBase(cbar_ax, cmap=COLORMAP, norm=norm, orientation="vertical")
        cb.set_label("log₂FC", fontsize=8, fontweight="bold")
        cb.ax.tick_params(labelsize=7)
        cb.outline.set_visible(True)
        cb.outline.set_linewidth(1.0)

        # Size legend (above the colorbar)
        size_ticks = [v for v in [1.0, 2.0, 3.0] if v <= global_max * 1.2]
        if not size_ticks:
            size_ticks = [0.5, 1.0]

        size_handles = [
            Line2D([0], [0], marker="o", color="w", markerfacecolor="#AAAAAA",
                   markeredgecolor="black", markeredgewidth=0.5,
                   markersize=np.sqrt(scale_size(v)) * 0.8,
                   label=f"{v:.1f}", lw=0)
            for v in size_ticks
        ]
        fig.legend(
            handles=size_handles, title="|log₂FC|",
            loc="upper left", bbox_to_anchor=(1.05, 0.98), # Pulled in tighter
            fontsize=7, title_fontproperties={"size": 8, "weight": "bold"},
            frameon=True, fancybox=True, edgecolor="#CCCCCC",
            labelspacing=1.0, handletextpad=0.8,
        )

        # Significance legend (below the colorbar)
        sig_handles = [
            Line2D([0], [0], marker="o", color="w", markerfacecolor="#AAAAAA",
                   markeredgecolor=BORDER_NS_COLOR, markeredgewidth=BORDER_NS_WIDTH,
                   markersize=10, lw=0, label="NS"),
            Line2D([0], [0], marker="o", color="w", markerfacecolor="#AAAAAA",
                   markeredgecolor=BORDER_FDR05_COLOR, markeredgewidth=BORDER_FDR05_WIDTH,
                   markersize=10, lw=0, label="FDR ≤ 0.05"),
            Line2D([0], [0], marker="o", color="w", markerfacecolor="#AAAAAA",
                   markeredgecolor=BORDER_FDR01_COLOR, markeredgewidth=BORDER_FDR01_WIDTH,
                   markersize=10, lw=0, label="FDR ≤ 0.01"),
        ]
        fig.legend(
            handles=sig_handles, title="Significance",
            loc="lower left", bbox_to_anchor=(1.05, 0.02), # Pulled in tighter
            fontsize=7, title_fontproperties={"size": 8, "weight": "bold"},
            frameon=True, fancybox=True, edgecolor="#CCCCCC",
            labelspacing=0.8,
        )

        # Save
        safe_ref  = REF_CONDITION.replace("-protocol", "")
        fname = f"bubble_{ent}_activation_vs_{safe_ref}"

        plt.savefig(os.path.join(RESULTS_DIR, f"{fname}.pdf"), dpi=DPI, bbox_inches="tight")
        plt.close()
        print(f"  Saved: {fname}.pdf")

print(f"\nDone. Results in: {RESULTS_DIR}")

Configuration loaded.
Reading from: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_3month/results/edgeR_raw
Filtered Entities for Plotting: ['FB']


findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because

Master CSV Saved: 22 rows for FB


findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because

  Saved: bubble_FB_activation_vs_3month.pdf

Done. Results in: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/fancy_bubbleplots/FB_activation
